# `runner.ipynb` — end-to-end Hindi LLM pretraining & SFT

Step-by-step runner for the whole pipeline in this repo:

**prepare corpus → train tokenizer → encode → pretrain (base) → SFT (chat) → evaluate → generate**

It drives the same scripts documented in the [README](../README.md); the notebook
just orchestrates them in order and adds inline inspection (param count, loss
curves, sample generations).

### Two modes
Set `MODE` in the config cell below:

- **`"smoke"`** — a tiny model on the tracked `data/sample_hindi.txt`. Runs on a
  laptop **CPU in well under a minute**. Output is gibberish by design — this only
  proves the wiring end-to-end.
- **`"real"`** — the full `configs/hindi_50m.yaml` (~50M params). **Put your Hindi
  corpus in `data/raw/` first.** Use a single **A100 / RTX 4090**; a meaningful run
  is hours, not seconds.

> Run the cells top to bottom. Each heavy step shells out to a script so the
> behavior is identical to running it from the terminal.

In [ ]:
# --- 0. Locate the repo root and make it the working directory -----------
# A notebook has no __file__, so we walk up from the current directory until we
# find the project markers (pyproject.toml + scripts/).
import os, sys, json, pathlib, shutil

def find_repo_root(start=None):
    p = pathlib.Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "pyproject.toml").exists() and (cand / "scripts").exists():
            return cand
    raise RuntimeError("Could not find repo root (pyproject.toml + scripts/).")

REPO = find_repo_root()
os.chdir(REPO)
# make `import hindi_llm` work in-process for the inspection cells
sys.path.insert(0, str(REPO / "src"))
print("repo root :", REPO)
print("cwd       :", os.getcwd())

In [ ]:
# --- 1. Environment check ------------------------------------------------
import torch
print("python  :", sys.version.split()[0])
print("torch   :", torch.__version__)
print("cuda?   :", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu/mps")
print("bf16?   :", torch.cuda.is_available() and torch.cuda.is_bf16_supported())
# Core deps are declared in pyproject. If something is missing, install the repo:
#   %pip install -e ".[dev]"           # core + pytest
#   %pip install -e ".[dev,demo,logging]"   # + gradio + wandb

In [ ]:
# --- 2. Configuration: choose MODE and derive all command arguments ------
MODE = "smoke"          # "smoke" (tiny, CPU, seconds) or "real" (full ~50M run)

CONFIG = "configs/hindi_50m.yaml"
TOKENIZER = "tokenizer/hindi_bpe.json"
PROCESSED = "data/processed"
BASE_DIR = "checkpoints/base"
SFT_DIR = "checkpoints/sft"
METRICS = "outputs/metrics.jsonl"

if MODE == "smoke":
    CORPUS_INPUT = "data/sample_hindi.txt"
    PREP_ARGS = f"--input {CORPUS_INPUT} --output-dir {PROCESSED} --txt-mode line --min-chars 80"
    TOK_ARGS = f"--input {PROCESSED}/clean.txt --output {TOKENIZER} --vocab-size 2000 --min-frequency 1"
    ENCODE_ARGS = f"--input {PROCESSED}/clean.jsonl --tokenizer {TOKENIZER} --val-fraction 0.1"
    # tiny model + short schedule via --set overrides
    MODEL_SET = ("--set model.context_length=64 --set model.d_model=128 "
                 "--set model.n_layers=2 --set model.n_heads=4")
    TRAIN_SET = ("--set train.batch_size=8 --set train.grad_accum_steps=1 "
                 "--set train.max_steps=60 --set train.eval_interval=20 "
                 "--set train.log_interval=10 --set train.sample_interval=0 "
                 "--set train.device=cpu")
    SFT_SET = ("--set sft.epochs=3 --set sft.batch_size=4 --set sft.warmup_steps=2 "
               "--set train.device=cpu")
elif MODE == "real":
    # >>> put your Hindi corpus files (.txt/.jsonl) in data/raw/ before running <<<
    CORPUS_INPUT = "data/raw"
    PREP_ARGS = f"--input {CORPUS_INPUT} --output-dir {PROCESSED} --min-chars 200 --min-devanagari 0.7"
    TOK_ARGS = f"--input {PROCESSED}/clean.txt --output {TOKENIZER} --vocab-size 32000 --min-frequency 2"
    ENCODE_ARGS = f"--input {PROCESSED}/clean.jsonl --tokenizer {TOKENIZER} --val-fraction 0.0005"
    MODEL_SET = ""                       # use configs/hindi_50m.yaml as-is
    TRAIN_SET = "--set train.device=auto"  # full schedule from the YAML
    SFT_SET = "--set train.device=auto"
else:
    raise ValueError("MODE must be 'smoke' or 'real'")

TRAIN_ARGS = f"--config {CONFIG} {MODEL_SET} {TRAIN_SET} --set checkpoint.out_dir={BASE_DIR}"
SFT_ARGS = (f"--config {CONFIG} {SFT_SET} "
            f"--set sft.data_path=data/sample_sft.jsonl "
            f"--set sft.base_checkpoint={BASE_DIR}/best.pt --set sft.out_dir={SFT_DIR}")
EVAL_ARGS = (f"--config {CONFIG} --base-checkpoint {BASE_DIR}/best.pt "
             f"--sft-checkpoint {SFT_DIR}/best.pt --out outputs/eval_report.md")

print("MODE         :", MODE)
print("corpus input :", CORPUS_INPUT)
print("base ckpt dir:", BASE_DIR)
print("sft  ckpt dir:", SFT_DIR)
if MODE == "real":
    print("\\n>>> Make sure your Hindi corpus is in data/raw/ before continuing. <<<")

## Step 1 — Prepare the corpus

Clean, filter, and deduplicate the raw text. Writes `clean.jsonl`, `clean.txt`,
`corpus_stats.json`, and a generated `data_card.md` under `data/processed/`.

In [ ]:
!python scripts/prepare_corpus.py {PREP_ARGS}

In [ ]:
# Peek at the generated stats + data card
import json
stats = json.load(open(f"{PROCESSED}/corpus_stats.json", encoding="utf-8"))
print("raw docs   :", stats["raw_docs"])
print("kept docs  :", stats["kept_docs"], f"({stats['kept_fraction']*100:.1f}%)")
print("total chars:", stats["total_chars"], "| est words:", stats["est_words"])
print("exact dups :", stats["exact_duplicates"], "| near dups:", stats["near_duplicates"])
print("\n--- data_card.md (head) ---")
print("\n".join(open(f"{PROCESSED}/data_card.md", encoding="utf-8").read().splitlines()[:24]))

## Step 2 — Train the Hindi BPE tokenizer

NFC → Metaspace → BPE. Writes the tokenizer JSON plus a report with fertility
(tokens/word), compression (chars/token), and unknown-token rate.

In [ ]:
!python scripts/train_tokenizer.py {TOK_ARGS}

In [ ]:
# Show the tokenizer report metrics + an example tokenization
rep = json.load(open(TOKENIZER.replace(".json", ".report.json"), encoding="utf-8"))
m = rep["metrics"]
print("vocab size :", rep["vocab_size"], "(requested", rep["requested_vocab"], ")")
print(f"fertility  : {m['fertility_tokens_per_word']:.3f} tokens/word")
print(f"compression: {m['compression_chars_per_token']:.3f} chars/token")
print(f"unknown    : {m['unknown_token_rate']:.5f}")

from hindi_llm.tokenizer_io import HindiTokenizer
tok = HindiTokenizer.load(TOKENIZER)
demo = "भारत एक विशाल और विविधताओं से भरा देश है।"
ids = tok.encode(demo)
print("\nexample    :", demo)
print("tokens     :", len(ids), "ids ->", ids[:20], "...")
print("roundtrip  :", tok.decode(ids))

## Step 3 — Encode the dataset into token shards

Tokenize the cleaned corpus into a flat token stream and split into
`train.bin` / `val.bin` (+ `meta.json`).

In [ ]:
!python scripts/encode_dataset.py {ENCODE_ARGS}

In [ ]:
meta = json.load(open(f"{PROCESSED}/meta.json", encoding="utf-8"))
print("dtype       :", meta["dtype"], "| vocab:", meta["vocab_size"])
print("total tokens:", meta["total_tokens"])
print("train tokens:", meta["train_tokens"], "| val tokens:", meta["val_tokens"])

## Step 4 — Inspect the model & parameter count

Build the configured model and confirm the parameter breakdown (this is the
"~50M" claim, verified by arithmetic that mirrors the modules).

In [ ]:
from hindi_llm.config import Config, print_param_count
cfg = Config.from_yaml(CONFIG)
cfg.sync_vocab(meta["vocab_size"])
# apply the same model overrides used for training (smoke shrinks the model)
if MODE == "smoke":
    cfg.model.context_length = 64
    cfg.model.d_model = 128
    cfg.model.n_layers = 2
    cfg.model.n_heads = 4
_ = print_param_count(cfg)

## Step 5 — Pretrain the base model

The hand-written training loop: bf16 autocast (GPU) / fp32 (CPU), gradient
accumulation, cosine LR + warmup, gradient clipping, periodic validation
perplexity, and `last`/`best` checkpointing.

> **Resume:** if a run is interrupted, re-run with `--resume` appended to
> `TRAIN_ARGS` (or set `checkpoint.resume=true`); it restores model + optimizer +
> step and continues the LR schedule from `checkpoints/base/last.pt`.

In [ ]:
# fresh metrics log so the loss-curve plot below is clean
import os
if os.path.exists(METRICS):
    os.remove(METRICS)
!python scripts/train.py {TRAIN_ARGS}

In [ ]:
# To CONTINUE an interrupted pretraining run, uncomment:
# !python scripts/train.py {TRAIN_ARGS} --resume

## Step 6 — Plot the loss curves

Read the always-on JSONL metrics log and plot train loss and validation loss.

In [ ]:
# read outputs/metrics.jsonl and plot; install matplotlib on demand
try:
    import matplotlib
except ImportError:
    %pip install -q matplotlib
    import matplotlib
import matplotlib.pyplot as plt

train_steps, train_loss = [], []
eval_steps, eval_loss, eval_ppl = [], [], []
for line in open(METRICS, encoding="utf-8"):
    r = json.loads(line)
    if "train_loss" in r:
        train_steps.append(r["step"]); train_loss.append(r["train_loss"])
    if "eval_val_loss" in r:
        eval_steps.append(r["step"]); eval_loss.append(r["eval_val_loss"]); eval_ppl.append(r["val_ppl"])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(train_steps, train_loss, label="train loss")
if eval_steps:
    ax[0].plot(eval_steps, eval_loss, "o-", label="val loss")
ax[0].set_xlabel("step"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("loss")
if eval_steps:
    ax[1].plot(eval_steps, eval_ppl, "o-", color="tab:red")
ax[1].set_xlabel("step"); ax[1].set_ylabel("val perplexity"); ax[1].set_title("validation perplexity")
plt.tight_layout(); plt.show()
if eval_ppl:
    print("final val perplexity:", round(eval_ppl[-1], 2))

## Step 7 — Generate from the base model (free-form)

Sanity check that the base model continues a Hindi prompt. In `smoke` mode this
is gibberish — that is expected.

In [ ]:
from hindi_llm.eval_utils import load_model_from_checkpoint, complete_prompts
from hindi_llm import train_utils as tu

device = tu.resolve_device("auto")
base_model, _ = load_model_from_checkpoint(f"{BASE_DIR}/best.pt", device)
for c in complete_prompts(base_model, tok,
                          ["भारत एक ऐसा देश है", "विज्ञान का महत्व यह है कि"],
                          device, max_new_tokens=40, temperature=0.8, top_k=40):
    print("prompt:", c["prompt"])
    print("output:", c["completion"])
    print("-" * 60)

## Step 8 — Supervised fine-tuning (SFT) → chat model

Fine-tune the base checkpoint on chat data using the Hindi chat template, with
loss masked to the assistant's tokens only. Produces a **separate** SFT
checkpoint. (Smoke mode uses the tiny tracked `data/sample_sft.jsonl`; for a real
run point `sft.data_path` at your own instruction data.)

In [ ]:
!python scripts/sft.py {SFT_ARGS}

## Step 9 — Evaluate (perplexity + qualitative report)

Computes deterministic validation perplexity for the base model and writes a
Markdown report with Hindi generations from both the base and SFT checkpoints,
plus honest failure-mode notes.

In [ ]:
!python scripts/evaluate.py {EVAL_ARGS}

In [ ]:
print("\n".join(open("outputs/eval_report.md", encoding="utf-8").read().splitlines()[:40]))

## Step 10 — Chat with the SFT model

In [ ]:
from hindi_llm.eval_utils import chat_responses
sft_model, _ = load_model_from_checkpoint(f"{SFT_DIR}/best.pt", device)
for c in chat_responses(sft_model, tok,
                        ["भारत की राजधानी क्या है?", "सूरज किस दिशा से उगता है?"],
                        device, max_new_tokens=64, temperature=0.7, top_k=40):
    print("user     :", c["user"])
    print("assistant:", c["assistant"])
    print("-" * 60)

## Step 11 — (Optional) Launch the Gradio chat demo

Run from a terminal (a blocking server is awkward inside a notebook):

```bash
python scripts/launch_gradio.py --checkpoint checkpoints/sft/best.pt
```

Requires the demo extra: `pip install -e ".[demo]"` (or `pip install gradio`).

---

### Done
- `smoke` mode just proves the pipeline runs end-to-end.
- For a **real** model: set `MODE = "real"`, put your corpus in `data/raw/`, point
  `sft.data_path` at your Hindi instruction data, and run on an A100 / RTX 4090.
  See [`docs/training_notes.md`](../docs/training_notes.md) for reading loss
  curves and diagnosing LR issues.